# Tabular analysis with DSLD
Explore adjusted group differences, possible confounders and proxies, frequency tables, and matching. Install the R package and `dsldPy` from this checkout first (see the [installation guide](https://github.com/matloff/dsld/blob/master/inst/README.md)). Functions accept pandas or R data frames. These examples use bundled data and reproducible subsamples.


In [ ]:
from dsldPy import (
    dsldPyLinear, dsldPyLinearSummary, dsldPyLinearPredict,
    dsldPyLogit, dsldPyLogitSummary, dsldPyLogitPredict, dsldPyML,
    dsldPyTakeALookAround, dsldPyCHunting, dsldPyOHunting,
    dsldPyFrequencybyS, dsldPyMatchedATE,
)


In [ ]:
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from dsldPy import dsld_Rpy2_RDataframeToPandas

ro.r["set.seed"](2026)
bundled = ro.Environment()
def load_bundled(name):
    importr("utils").data(name, package="dsld", envir=bundled)
    return dsld_Rpy2_RDataframeToPandas(bundled[name])

census = load_bundled("svcensus").sample(n=600, random_state=2026)
wages = census[["age", "wkswrkd", "gender", "wageinc"]]
wage_points = wages[["age", "wkswrkd"]].head(2)
records = load_bundled("compas1")
records = records.loc[records["race"].isin(["Caucasian", "African-American"])]
records = records[["age", "priors_count", "race", "two_year_recid"]].sample(n=600, random_state=2026)
records["race"] = records["race"].cat.remove_unused_categories()
record_points = records[["age", "priors_count"]].head(2)


## 1. Adjusted group comparisons
An interaction model fits a separate regression for each sensitive group. Comparison points omit the outcome and sensitive columns. For a model without interactions, prediction inputs include the sensitive column.


In [ ]:
linear_model = dsldPyLinear(wages, "wageinc", "gender", interactions=True,
                             sComparisonPts=wage_points)
dsldPyLinearSummary(linear_model)
dsldPyLinearPredict(linear_model, wage_points)


In [ ]:
logit_model = dsldPyLogit(records, "two_year_recid", "race",
                          sComparisonPts=record_points, interactions=True, yesYVal="Yes")
dsldPyLogitSummary(logit_model)
dsldPyLogitPredict(logit_model, record_points)


In [ ]:
errors_by_group, comparisons = dsldPyML(
    wages, "wageinc", "gender", "qeKNN", sComparisonPts=wage_points,
    opts={"k": 10, "scaleX": True, "holdout": 30},
)
print(errors_by_group)
comparisons


## 2. Compare feature sets
Columns `a` and `b` are errors for predicting Y without and with S. Column `c` is the error for predicting S: a low value may indicate strong proxy information. All feature sets use the same holdout rows.


In [ ]:
dsldPyTakeALookAround(wages, "wageinc", "gender", maxFeatureSize=2, holdout=100)


## 3. Explore confounders and proxies
Importance rankings and correlations are exploratory aids; they do not establish causal adjustment sets.


In [ ]:
confounders = dsldPyCHunting(wages, "wageinc", "gender", intersectDepth=2)
print(confounders)
dsldPyOHunting(wages, "wageinc", "gender")


## 4. Group frequencies


In [ ]:
dsldPyFrequencybyS(census, "educ", "gender")


## 5. Matching-based estimates
Specify both the treatment/exposure label and the positive outcome label. A causal interpretation requires assumptions about the selected covariates and study design.


In [ ]:
matched = dsldPyMatchedATE(records, "two_year_recid", "race",
                           yesSVal="Caucasian", yesYVal="Yes")
list(matched.rx2("est"))
